# Agribusiness Data Analysis Planning - Week 1
## Notebook 01: Data Source Exploration, Evaluation & Comparison Matrix

### Project Overview
This notebook presents the research, documentation, and technical evaluation of publicly available agricultural datasets and APIs. In accordance with data science best practices, we evaluate institutional provenance, temporal/spatial granularity, update cadence, access protocols, and data quality limitations.

> **Disclaimer**: Week 1 focuses on technical evaluation, API testing, and analytical planning. No models or empirical forecasts have been trained yet.

In [ ]:
import os
import sys
from pathlib import Path
import requests
import pandas as pd
import numpy as np

# Add repository root to system path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.utils import setup_logger
logger = setup_logger()
print("Environment initialized successfully. Repository root:", repo_root)

### 1. Data Source Catalog & Technical Specifications

Below is the consolidated matrix of candidate data sources evaluated for our agribusiness analytics framework:

In [ ]:
sources_data = [
    {
        "Source Identifier": "FAOSTAT-QCL",
        "Organization": "Food and Agriculture Organization (FAO), UN",
        "Portal / Endpoint URL": "https://www.fao.org/faostat/en/#data/QCL",
        "Geographic Coverage": "Global (245 countries and territories)",
        "Temporal Coverage": "1961 - Present (Annual)",
        "Key Variables": "Crops, Area Harvested (ha), Production (tonnes), Yield (kg/ha)",
        "Data Format / API": "REST API, Bulk CSV, JSON",
        "Primary Analytical Use": "Cross-national yield comparisons, long-term global production trajectories",
        "Known Limitations": "2-year publication lag; country-level aggregation only (no sub-national/district data)"
    },
    {
        "Source Identifier": "WORLD-BANK-WDI",
        "Organization": "The World Bank Group",
        "Portal / Endpoint URL": "https://data.worldbank.org/indicator",
        "Geographic Coverage": "Global (217 economies)",
        "Temporal Coverage": "1960 - Present (Annual)",
        "Key Variables": "Cereal yield, Ag land (% of land), Ag value added (% of GDP), Employment",
        "Data Format / API": "REST API (v2), JSON, XML, CSV",
        "Primary Analytical Use": "Macroeconomic context, agricultural transformation, resource utilization indicators",
        "Known Limitations": "Macro-aggregated; lacks commodity-specific price series or subnational granularity"
    },
    {
        "Source Identifier": "INDIA-APY",
        "Organization": "Directorate of Economics & Statistics (DES) / data.gov.in",
        "Portal / Endpoint URL": "https://data.gov.in/resource/crop-production-statistics",
        "Geographic Coverage": "India (36 States/UTs, 700+ Districts)",
        "Temporal Coverage": "1997 - 2021+ (Seasonal / Annual)",
        "Key Variables": "State, District, Crop Year, Season, Crop, Area (ha), Production (tonnes)",
        "Data Format / API": "OGD India REST API / CSV bulk downloads",
        "Primary Analytical Use": "District-level spatial productivity, seasonal crop portfolio analysis, yield growth",
        "Known Limitations": "Historical district boundary changes; missing values for newly carved districts"
    },
    {
        "Source Identifier": "AGMARKNET",
        "Organization": "Directorate of Marketing & Inspection (DMI), MoA&FW, GoI",
        "Portal / Endpoint URL": "https://agmarknet.gov.in/",
        "Geographic Coverage": "India (3,200+ APMC Mandis)",
        "Temporal Coverage": "2001 - Present (Daily arrivals and prices)",
        "Key Variables": "Market, Commodity, Variety, Arrival Date, Min Price, Max Price, Modal Price (INR/q)",
        "Data Format / API": "Web Portal tables, data.gov.in API",
        "Primary Analytical Use": "Price volatility modeling, market integration, seasonality, time-series forecasting",
        "Known Limitations": "Intermittent reporting by smaller mandis; holiday gaps requiring imputation"
    },
    {
        "Source Identifier": "USDA-PSD",
        "Organization": "Foreign Agricultural Service (FAS), USDA",
        "Portal / Endpoint URL": "https://apps.fas.usda.gov/psdonline/app/index.html",
        "Geographic Coverage": "Global Commodity Level (Major producing/trading nations)",
        "Temporal Coverage": "1960 - Present (Annual / Monthly revisions)",
        "Key Variables": "Beginning stocks, Production, Imports, Domestic consumption, Exports, Ending stocks",
        "Data Format / API": "CSV download, FAS API",
        "Primary Analytical Use": "Global supply-demand balance, stock-to-use ratios, international price transmission",
        "Known Limitations": "Focuses primarily on grains, oilseeds, and cotton; excludes minor regional crops"
    }
]

df_sources = pd.DataFrame(sources_data)
df_sources[["Source Identifier", "Organization", "Geographic Coverage", "Temporal Coverage", "Data Format / API"]]

### 2. Live API Connectivity & Schema Verification Test
Let's verify the programmatic connectivity to the World Bank Open Data API for standard agricultural indicators.

In [ ]:
# Test connectivity to World Bank API
test_url = "https://api.worldbank.org/v2/country/IND/indicator/AG.YLD.CREL.KG?date=2015:2022&format=json"
try:
    res = requests.get(test_url, timeout=15)
    res.raise_for_status()
    data = res.json()
    print(f"API Connection Status: HTTP {res.status_code}")
    print(f"Returned Metadata: Total {data[0]['total']} records across {data[0]['pages']} page(s)")
    
    # Sample record structure
    sample = data[1][0]
    print("\nSample Record Payload:")
    for k, v in sample.items():
        print(f"  - {k}: {v}")
except Exception as e:
    print("API request failed or restricted in environment:", e)

### 3. Summary of Research Findings & Proposed Scope for Week 2 Ingestion
- **Primary Panel**: India APY district-level dataset (1997-2021) covers over 70 crops across 700 districts, ideal for regional productivity, crop yield comparisons, and cropping intensity analysis.
- **Commodity Price Panel**: Agmarknet APMC mandi prices for key staple commodities (Wheat, Rice, Chickpea, Mustard, Onion, Potato) to model wholesale price volatility and seasonal variations.
- **Macroeconomic Baseline**: World Bank Cereal Yield & Ag GDP % indicators to provide international and cross-decadal comparative context.